# Links

GitHub repository:
https://github.com/nylandgit/IND320-Repo

Streamlit app:
https://nylandgit-ind320-repo.streamlit.app/

Code runthrough video:


# AI Statement and Descriptive Log

## AI Statement

This project has used the free VS Code Copilot generative AI to: Generate code, identify bugs, fix bugs, brainstorm technical solutions, and to teach the process of coding through in-depth explanations of each generated line of code. Some Google searches have also involved Google's built-in generative AI.

The project author has steered the AI's coding through accepting, rejecting, and modifying the AI's code based on the author's intuition. The author has requested code explanations, studied the AI's code, and made explanatory comments for each code block based on the author's understanding.

The code provided is therefore created partly by the author and partly by generative AI. The explanatory code comments are written exclusively by the author.

Anthropic's Claude was used to develop the Open-Meteo API data retreival code.

## Descriptive Log

### To-Do For Part 4 (When all comes together! :) )
* Clean up Streamlit app, improve user experience, make dashboard coherent and intuitive.
* Improve quality of plots and graphs. Create explanations where relevant.
* Consider sentralizing the taking of user input in side panel to show their effects across various plots, rather than spread user inputs over multiple pages. Generally break free of the task text design and create something better.
* Optimize Streamlit pages' resource usage, make the app run faster and smoother with reduced wait times.
* Clean up GitHub repository. Remove unneeded and mistakenly tracked files. Add ReadMe and explanation of file structure.
* Create updated requirements.txt, or alternatively uv.lock.
* Finalize Streamlit app and GitHub repo so they are presentable, usable, and as good as they can be.

<p></p>
<p></p>

### Section 1:
* Created Norges Bank API retrieval function currency_download(), used custom url from Norges Bank website based on relevant data filters. Tested function.
* Investigated timezone differences across API data. See section 1.2 for conclusions.

<p></p>
<p></p>

### Streamlit
* Text

# Non-Streamlit Code

## 1. Norges Bank API, API Timezones

### 1.1 Currency Exchange Rates Function

In [11]:
import pandas as pd
import requests

# Creating function as instructed.
def currency_download():

    # Defining url copied from NB API website where custom urls can be made.
    url = "https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP?endPeriod=2026-11-18&format=sdmx-json&locale=en&startPeriod=1999-01-04"

    # Retrieving data from API with .get().
    response = requests.get(url, timeout=45)
    response.raise_for_status()

    # The API returns requested data according to the SDMX-JSON standard, as
    # indicated by the url. To correctly extract the data to a pandas df, the
    # response needs to be decoded. .json() parses the HTTP response into
    # Python lists and dictionaries.
    nb_response = response.json()

    # Selecting dataset and defining dimensions and dates for soon-to-be df.
    dataset = nb_response["data"]["dataSets"][0]
    dimensions = nb_response["data"]["structure"]["dimensions"]
    dates = dimensions["observation"][0]["values"]

    # Iterating over each currency series in dataset to populate rows with data
    # for soon-to-be df.
    rows = []
    for key, series in dataset["series"].items():
        
        # Mapping data (?).
        indexes = [int(i) for i in key.split(":")]
        labels = {
            dimension["id"]: dimension["values"][i]["id"]
            for dimension, i in zip(dimensions["series"], indexes)
        }

        # Iterating over each observation (data point) in each series.
        for period_index, observation in series["observations"].items():
            
            # Adding data to rows for soon-to-be df.
            rows.append({
                "date": dates[int(period_index)]["id"],
                "currency": labels["BASE_CUR"],
                "rate": float(observation[0]),
            })

    # Creating df based on data in rows.
    df = pd.DataFrame(rows).pivot(
        index="date", columns="currency", values="rate")
    
    df.columns = [f"{currency}_rate" for currency in df.columns]
    df = df.reset_index()[["date", "USD_rate", "EUR_rate"]]

    #print(df)
    return df

df_retrieved = currency_download()
print(df_retrieved)


            date  USD_rate  EUR_rate
0     1999-01-04    7.5112    8.8550
1     1999-01-05    7.4378    8.7745
2     1999-01-06    7.4326    8.7335
3     1999-01-07    7.4142    8.6295
4     1999-01-08    7.3677    8.5900
...          ...       ...       ...
7018  2026-11-12    9.5325   10.5630
7019  2026-11-13    9.5634   10.5695
7020  2026-11-16    9.6005   10.5740
7021  2026-11-17    9.6237   10.5755
7022  2026-11-18    9.6686   10.6045

[7023 rows x 3 columns]


### 1.2 API Timezone Investigation and Conclusion

The task was completed by prompting GitHub copilot to inspect all relevant files calling the APIs and/or data from them. The results have been compiled by the author.

Weekly/daily observations:

* NVE Magasinstatistikk: Weekly observations. Timezone-naive, the retrieved data does not include timezone specifications. Timestamps for observations are treated as dates, Norwegian timezone assumed if relevant.
* Norges Bank: Daily observations, timezone-naive. Timestamps for observations are treated as dates, Norwegian timezone assumed if relevant.

<p></p>

Hourly observations:

* ENTSO-E: Hourly observations. Timezone-aware, called by entsoe-py in Section 3.2 of project_work_part2.ipynb. The time data is converted to UTC+0, which in turn has been placed into Cassandra and MongoDB.
* Open-Meteo: Hourly observations. Timezone-aware, UTC "Europe/Oslo", i.e. UTC+1 for winter time and UTC+2 for summer time.

Conclusion:

ENTSO-E and Open-Meteo use hourly observations, but Open-Meteo is 1 hour ahead of ENTSO-E in winter time and 2 hours ahead in summer time.

NVE Magasinstatistikk and Norges Bank use weekly and daily observations (respectively).

For cross-API comparisons of data points, a UTC conversion may be necessary. The author will investigate which API data must be converted to fit the other(s) on a case-by-case basis. ENTSO-E data has been placed into Cassandra and MongoDB, so it may be preferable to avoid changes to this data. However, as the three other APIs all use Norwegian timezones, it may be beneficial to make only one change to ENTSO-E, rather than three changes to the other APIs.

### 1.3 Sampling of APIs (not necessary, used in investigation)

In [ ]:
# This code is AI-derived, the author has not spent much time on it, hence
# the lack of comments.
import os
from datetime import timedelta

import pandas as pd
import requests
import openmeteo_requests
from retry_requests import retry
from entsoe import EntsoePandasClient

def api_time_sample():
    """Print four observations from each of the four APIs."""

    print("\n=== MAGASINSTATISTIKK ===")
    mag_url = (
        "https://biapi.nve.no/magasinstatistikk/api/"
        "Magasinstatistikk/HentOffentligData"
    )
    mag_response = requests.get(mag_url, timeout=45)
    mag_response.raise_for_status()

    mag = pd.DataFrame(mag_response.json())
    mag = mag.rename(columns={
        "dato_Id": "Date",
        "neste_Publiseringsdato": "Next Publishing Date",
        "omrType": "Area Type",
        "omrnr": "Area Number",
    })
    mag["Date"] = pd.to_datetime(mag["Date"], errors="coerce")

    print(mag[[
        "Date",
        "Area Type",
        "Area Number",
        "fyllingsgrad",
    ]].head(4).to_string(index=False))

    print("\n=== ENTSO-E ===")
    api_key = os.getenv("ENTSO_API_KEY")

    if not api_key:
        print("ENTSO_API_KEY environment variable is not configured.")
    else:
        entsoe_client = EntsoePandasClient(api_key=api_key)
        end = pd.Timestamp.now(tz="UTC").floor("h")
        start = end - timedelta(days=7)

        entsoe_series = entsoe_client.query_crossborder_flows(
            "NO_1",
            "SE_3",
            start=start,
            end=end,
        )

        entsoe = entsoe_series.rename(
            "flow_mw"
        ).rename_axis("timestamp").reset_index()

        entsoe["timestamp"] = pd.to_datetime(
            entsoe["timestamp"],
            utc=True,
        )

        print(entsoe.head(4).to_string(index=False))

    print("\n=== OPEN-METEO ===")
    retry_session = retry(
        requests.Session(),
        retries=5,
        backoff_factor=0.2,
    )
    openmeteo = openmeteo_requests.Client(session=retry_session)

    weather_url = "https://archive-api.open-meteo.com/v1/archive"
    parameters = {
        "latitude": 59.91,
        "longitude": 10.75,
        "start_date": "2024-01-01",
        "end_date": "2024-01-02",
        "hourly": ["temperature_2m", "precipitation"],
        "timezone": "Europe/Oslo",
    }

    responses = openmeteo.weather_api(
        weather_url,
        params=parameters,
    )
    response = responses[0]
    hourly = response.Hourly()

    weather = pd.DataFrame({
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        ).tz_convert("Europe/Oslo"),
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "precipitation": hourly.Variables(1).ValuesAsNumpy(),
    })
    weather["UTC"] = weather["date"].dt.tz_convert("UTC")

    print(weather.head(4).to_string(index=False))

    print("\n=== NORGES BANK ===")
    nb_url = (
        "https://data.norges-bank.no/api/data/"
        "EXR/B.USD+EUR.NOK.SP"
        "?startPeriod=2024-01-01"
        "&endPeriod=2024-01-10"
        "&format=sdmx-json"
        "&locale=en"
    )
    nb_response = requests.get(nb_url, timeout=45)
    nb_response.raise_for_status()

    nb_json = nb_response.json()
    dataset = nb_json["data"]["dataSets"][0]

    dates = (
        nb_json["data"]["structure"]["dimensions"]["observation"][0]
        ["values"]
    )
    series_dimensions = (
        nb_json["data"]["structure"]["dimensions"]["series"]
    )
    currency_values = series_dimensions[1]["values"]

    rows = []
    for key, series in dataset["series"].items():
        currency_index = int(key.split(":")[1])
        currency = currency_values[currency_index]["id"]

        for position, observation in series["observations"].items():
            rows.append({
                "date": dates[int(position)]["id"],
                "currency": currency,
                "rate": float(observation[0]),
            })

    nb = pd.DataFrame(rows).pivot(
        index="date",
        columns="currency",
        values="rate",
    ).reset_index()

    print(nb.head(4).to_string(index=False))

api_time_sample()



=== MAGASINSTATISTIKK ===
      Date Area Type  Area Number  fyllingsgrad
2020-11-15        EL            1      0.974361
2011-08-28        EL            4      0.786499
1998-07-05        EL            3      0.793969
2011-03-27        EL            4      0.300820

=== ENTSO-E ===
ENTSO_API_KEY environment variable is not configured.

=== OPEN-METEO ===
                     date  temperature_2m  precipitation                       UTC
2024-01-01 00:00:00+01:00         -4.0805            0.2 2023-12-31 23:00:00+00:00
2024-01-01 01:00:00+01:00         -3.9805            0.1 2024-01-01 00:00:00+00:00
2024-01-01 02:00:00+01:00         -3.7305            0.9 2024-01-01 01:00:00+00:00
2024-01-01 03:00:00+01:00         -4.0305            0.6 2024-01-01 02:00:00+00:00

=== NORGES BANK ===
      date     EUR     USD
2024-01-02 11.2815 10.2971
2024-01-03 11.3200 10.3672
2024-01-04 11.2845 10.3027
2024-01-05 11.3090 10.3553


## 2. Machine Learning

### 2.1 WIP